# Task 1 — Define Source and Extract

# Pluralsight Content Extraction

## Objective

This notebook extracts educational article data from Pluralsight's public AI & Data and Cloud blog categories.

The extracted data will be saved as raw JSON files and used as the input for Task 2 — Data Discovery, Profiling and Cleaning.

### Data Sources

1. Pluralsight AI & Data
   https://www.pluralsight.com/resources/blog/ai-and-data

2. Pluralsight Cloud
   https://www.pluralsight.com/resources/blog/cloud

The extracted metadata includes:

- Source
- Category
- Title
- Author
- Publication date
- Description
- Tags
- URL
- Article content
- Scraped timestamp

## Source Definition

### Source
Pluralsight public technology blog.

### Authentication
No authentication or API key is required for the public blog pages.

### Collection Method
The project uses Python HTTP requests and BeautifulSoup to retrieve and parse publicly accessible HTML pages.

### Categories
- AI & Data
- Cloud

### Pagination
The category pages are paginated. The scraper follows category pages and extracts individual article URLs.

### Rate Limiting
A delay of 2 seconds is applied between requests to reduce request frequency.

HTTP 429 responses are handled by waiting before retrying.

### Terms and Usage
The project is an educational data engineering project. Only publicly available article content and metadata are collected, and the scraper uses a low request rate.

In [10]:
# Imports
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import json
import time
import os
from datetime import datetime

In [11]:
# Configuration
BASE_URL = "https://www.pluralsight.com"

CATEGORIES = {
    "AI & Data": {
        "url": "https://www.pluralsight.com/resources/blog/ai-and-data",
        "output": "data/raw/pluralsight_ai_data_articles.json",
        "max_pages": 31
    },
    "Cloud": {
        "url": "https://www.pluralsight.com/resources/blog/cloud",
        "output": "data/raw/pluralsight_cloud_articles.json",
        "max_pages": 47
    }
}

MAX_ARTICLES = 10 # change this after trial #######################################
REQUEST_DELAY = 2

USER_AGENT = "learningProjectPipeline/1.0"

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Language": "en-US,en;q=0.9"
}

session = requests.Session()
session.headers.update(HEADERS)

## HTTP Request Function

The following function retrieves a webpage and handles common HTTP errors.

A delay is applied after each request to reduce the request rate.

In [12]:
def fetch_page(url, retries=3):

    for attempt in range(retries):

        try:
            response = session.get(url, timeout=30)

            print(f"GET {url} -> {response.status_code}")

            if response.status_code == 200:
                time.sleep(REQUEST_DELAY)
                return response.text

            elif response.status_code == 429:
                print("Rate limited. Waiting before retrying...")
                time.sleep(10)

            elif response.status_code >= 500:
                print("Server error. Retrying...")
                time.sleep(5)

            else:
                print(f"Request failed with status {response.status_code}")
                return None

        except requests.RequestException as e:
            print(f"Request error: {e}")
            time.sleep(5)

    return None

## URL Normalization

Article URLs are normalized by removing query parameters and fragments.

This prevents the same article from being stored multiple times when it appears with different tracking parameters.

In [13]:
def normalize_url(url):

    parsed = urlparse(url)

    return parsed._replace(
        query="",
        fragment=""
    ).geturl()

## Article URL Discovery

The category pages are scanned to identify individual article URLs.

Only URLs belonging to the selected Pluralsight category are retained.

In [14]:
def extract_article_links(html, category_url):

    soup = BeautifulSoup(html, "html.parser")

    links = set()

    category_path = urlparse(category_url).path.rstrip("/") + "/"

    for a in soup.find_all("a", href=True):

        url = urljoin(BASE_URL, a["href"])
        url = normalize_url(url)

        parsed_path = urlparse(url).path

        if parsed_path.startswith(category_path):
            links.add(url)

    return links

In [15]:
def discover_article_urls(category_url, max_pages):

    article_urls = set()

    for page_number in range(1, max_pages + 1):

        if page_number == 1:
            page_url = category_url
        else:
            page_url = f"{category_url}?page={page_number}"

        print(f"\nScanning page {page_number}: {page_url}")

        html = fetch_page(page_url)

        if not html:
            continue

        links = extract_article_links(
            html,
            category_url
        )

        print(f"Found {len(links)} article links")

        article_urls.update(links)

    return sorted(article_urls)

## Article Metadata Extraction

Each article page is parsed to extract standardized metadata.

The extraction uses HTML selectors specific to the Pluralsight article structure.

The Table of Contents, navigation elements, scripts, styles, promotional content, and author biography are removed before article content is extracted.

In [16]:
def extract_article(url, html, category):

    soup = BeautifulSoup(html, "html.parser")

    # --------------------------------------------------
    # REMOVE TABLE OF CONTENTS
    # --------------------------------------------------

    toc = soup.find("div", class_="table-of-contents")

    if toc:
        toc.decompose()

    # --------------------------------------------------
    # REMOVE PAGE ELEMENTS THAT ARE NOT ARTICLE CONTENT
    # --------------------------------------------------

    for element in soup.find_all(
        ["nav", "header", "footer", "aside", "script", "style"]
    ):
        element.decompose()

    # --------------------------------------------------
    # TITLE
    # --------------------------------------------------

    title = ""

    title_element = soup.find("h1")

    if title_element:
        title = title_element.get_text(" ", strip=True)

    # --------------------------------------------------
    # DESCRIPTION
    # --------------------------------------------------

    description = ""

    meta_description = soup.find(
        "meta",
        attrs={"name": "description"}
    )

    if meta_description:
        description = meta_description.get("content", "").strip()

    # --------------------------------------------------
    # TAGS
    # --------------------------------------------------

    tags = []

    tag_list = soup.find(
        "ul",
        class_="tag-list-listing"
    )

    if tag_list:

        tags = [
            li.get_text(" ", strip=True)
            for li in tag_list.find_all("li")
            if li.get_text(strip=True)
        ]

    # --------------------------------------------------
    # AUTHOR
    # --------------------------------------------------

    author = ""

    author_meta = soup.find(
        "meta",
        attrs={"name": "author"}
    )

    if author_meta:
        author = author_meta.get("content", "").strip()

    if not author:

        for element in soup.find_all(["p", "div", "span"]):

            text = element.get_text(" ", strip=True)

            if (
                text.startswith("By ")
                and len(text) < 100
            ):
                author = text[3:].strip()
                break

    # --------------------------------------------------
    # PUBLICATION DATE
    # --------------------------------------------------

    publication_date = ""

    date_element = soup.find(
        "p",
        class_="date-length-text"
    )

    if date_element:

        text = date_element.get_text(
            " ",
            strip=True
        )

        publication_date = text.split("•")[0].strip()

    # --------------------------------------------------
    # ARTICLE CONTAINER
    # --------------------------------------------------

    article = soup.find("article")

    if not article:
        article = soup.find("main")

    content_parts = []

    if article:

        # Remove unwanted CTA / biography sections
        for element in article.find_all(
            ["div", "section", "aside"]
        ):

            text = element.get_text(
                " ",
                strip=True
            )

            if (
                "Advance your tech skills today" in text
                or "is a seasoned" in text
            ):
                element.decompose()

        # --------------------------------------------------
        # ARTICLE CONTENT IN DOM ORDER
        # --------------------------------------------------

        for element in article.find_all(
            ["h1", "h2", "h3", "h4", "p", "li"]
        ):

            text = element.get_text(
                " ",
                strip=True
            )

            if not text:
                continue

            if text.startswith("By "):
                continue

            if "Minute Read" in text:
                continue

            if element.name == "li":

                if element.find_parent(
                    "ul",
                    class_="tag-list-listing"
                ):
                    continue

                content_parts.append("- " + text)

            elif element.name in ["h1", "h2", "h3", "h4"]:

                content_parts.append(
                    "\n" + text + "\n"
                )

            else:

                content_parts.append(text)

    content = "\n\n".join(content_parts).strip()

    # Final cleanup boundary
    if "Advance your tech skills today" in content:

        content = content.split(
            "Advance your tech skills today",
            1
        )[0].strip()

    # --------------------------------------------------
    # RETURN STANDARDIZED RECORD
    # --------------------------------------------------

    return {
        "source": "Pluralsight",
        "category": category,
        "title": title,
        "author": author,
        "publication_date": publication_date,
        "description": description,
        "tags": tags,
        "url": url,
        "content": content,
        "scraped_at": datetime.now().isoformat()
    }

In [17]:
def scrape_category(category_name, category_config):

    output_file = category_config["output"]

    os.makedirs(
        os.path.dirname(output_file),
        exist_ok=True
    )

    # Load existing records for incremental scraping
    if os.path.exists(output_file):

        with open(
            output_file,
            "r",
            encoding="utf-8"
        ) as f:

            records = json.load(f)

    else:
        records = []

    existing_urls = {
        record["url"]
        for record in records
        if "url" in record
    }

    print(
        f"\nExisting {category_name} records: "
        f"{len(existing_urls)}"
    )

    # Discover URLs
    article_urls = discover_article_urls(
        category_config["url"],
        category_config["max_pages"]
    )

    print(
        f"Total discovered URLs: "
        f"{len(article_urls)}"
    )

    # Incremental extraction
    for url in article_urls:

        if len(records) >= MAX_ARTICLES:
            break

        if url in existing_urls:
            continue

        print(f"\nScraping article: {url}")

        html = fetch_page(url)

        if not html:
            continue

        try:

            record = extract_article(
                url,
                html,
                category_name
            )

            records.append(record)

            existing_urls.add(url)

            with open(
                output_file,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    records,
                    f,
                    ensure_ascii=False,
                    indent=2
                )

            print(
                f"Saved: {record['title']}"
            )

        except Exception as e:

            print(
                f"Extraction error: {e}"
            )

    return records

In [18]:
all_results = {}

for category_name, category_config in CATEGORIES.items():

    print("=" * 60)
    print(f"SCRAPING: {category_name}")
    print("=" * 60)

    results = scrape_category(
        category_name,
        category_config
    )

    all_results[category_name] = results

    print(
        f"\n{category_name}: "
        f"{len(results)} records"
    )

SCRAPING: AI & Data

Existing AI & Data records: 0

Scanning page 1: https://www.pluralsight.com/resources/blog/ai-and-data
GET https://www.pluralsight.com/resources/blog/ai-and-data -> 200
Found 4 article links

Scanning page 2: https://www.pluralsight.com/resources/blog/ai-and-data?page=2
GET https://www.pluralsight.com/resources/blog/ai-and-data?page=2 -> 200
Found 7 article links

Scanning page 3: https://www.pluralsight.com/resources/blog/ai-and-data?page=3
GET https://www.pluralsight.com/resources/blog/ai-and-data?page=3 -> 200
Found 7 article links

Scanning page 4: https://www.pluralsight.com/resources/blog/ai-and-data?page=4
GET https://www.pluralsight.com/resources/blog/ai-and-data?page=4 -> 200
Found 8 article links

Scanning page 5: https://www.pluralsight.com/resources/blog/ai-and-data?page=5
GET https://www.pluralsight.com/resources/blog/ai-and-data?page=5 -> 200
Found 4 article links

Scanning page 6: https://www.pluralsight.com/resources/blog/ai-and-data?page=6
GET http

## Extraction Results

The extracted records are stored as raw JSON files under `data/raw/`.

The files produced by this notebook are:

- `data/raw/pluralsight_ai_data_articles.json`
- `data/raw/pluralsight_cloud_articles.json`

These files serve as the input to Task 2.